# Misclassification Analysis from soft-vote Ensemble Model

In [15]:
# imports
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb # type: ignore

# helpers
sys.path.insert(0, "../../")

from _settings import LABEL_COLS
from models_helper import (
    analyze_rf_misses,
    analyze_xgb_misses,
    find_best_model_path,
    load_feature_names,
    load_model_pickle,
    plot_xgb_misses_contributions
)

In [16]:
# define directories and paths
MISCLASS_DIR = Path("../5_3_soft_vote_ensemble_model/misclassifications_per_project")
MODELS_DIR = "../../4_modeling/4_5_hyperparameter_tuning/models"
BEST_RUNS_PATH = "../../5_evaluation/5_1_get_best_runs_wandb/best_by_model_type.json"

# output path for the structured analysis results consumed by the report
RESULTS_PATH = "misclassification_analysis_results.json"

# accumulator: every analyze_*_misses call below stores its per-row results here
results = {"rf": {col: {} for col in LABEL_COLS}, "xgboost": {col: {} for col in LABEL_COLS}}

## 1. Load the misclassification CSVs from models RF and XGBoost from test set evaluation

In [17]:
# load all per-project misclassification CSVs into one DataFrame
csv_paths  = sorted(MISCLASS_DIR.glob("*_misclassifications.csv"))
full_table = pd.concat([pd.read_csv(p) for p in csv_paths], ignore_index=True)

print(f"Loaded {len(full_table):,} misclassified rows from {len(csv_paths)} project CSVs")
print(f"Projects: {full_table['project_code'].unique().tolist()}")

Loaded 2,169 misclassified rows from 2 project CSVs
Projects: ['ADEM', 'LUMU']


## 2. Pick an example misclassification with the label to inspect

In [18]:
# example label to inspect
EXAMPLE_LABEL = "label_predefined_type"

# candidates: rows where the soft-vote ensemble itself got the target label wrong
true_col     = f"{EXAMPLE_LABEL}__true"
ensemble_col = f"{EXAMPLE_LABEL}__ensemble_pred"
candidates   = full_table[full_table[ensemble_col] != full_table[true_col]].reset_index(drop=True)

# get one example row to inspect
EXAMPLE_IDX = 1
example     = candidates.iloc[EXAMPLE_IDX]

print(f"Project: {example['project_code']}")
print(f"GUID: {example['guid']}")
print(f"Label: {EXAMPLE_LABEL}\n")

print(f"True class: {example[f'{EXAMPLE_LABEL}__true']}")
print(f"Ensemble predicted: {example[f'{EXAMPLE_LABEL}__ensemble_pred']}")
print(f"Voted by: {example[f'{EXAMPLE_LABEL}__ensemble_voted_by']}\n")

print(f"RF predicted: {example[f'{EXAMPLE_LABEL}__rf_pred']}")
print(f"XGB predicted: {example[f'{EXAMPLE_LABEL}__xgboost_pred']}")
print(f"NN predicted: {example[f'{EXAMPLE_LABEL}__nn_pred']}")

Project: ADEM
GUID: 2xVIknWIj6Nepc1hRyicEE
Label: label_predefined_type

True class: COLUMN
Ensemble predicted: NOTDEFINED
Voted by: rf|nn

RF predicted: NOTDEFINED
XGB predicted: SOLIDWALL
NN predicted: NOTDEFINED


## 3. Load RF model for later analysis

In [19]:
# load RF model
rf_path = find_best_model_path("rf", MODELS_DIR)
rf_full = load_model_pickle(rf_path)

rf = rf_full.pipeline.named_steps["model"]
rf_features = load_feature_names("rf", BEST_RUNS_PATH)

print(f"Loaded RF ({len(rf.estimators_)} trees) from {rf_path.name}")
print(f"RF was trained on {len(rf_features)} features")

Loaded RF (200 trees) from best_rf_model_0.7143.pkl
RF was trained on 37 features


### 3.1. Analyse label "ifc_entity" and target class "IfcPlate"

In [20]:
results["rf"]["label_ifc_entity"]["IfcPlate"] = analyze_rf_misses(rf, full_table, rf_features, "label_ifc_entity", "IfcPlate", LABEL_COLS)

Found 26 rows where RF misclassified true class 'IfcPlate' on label_ifc_entity

[0] ADEM / 1UuRmxrOH3IfBQHKvkX3vv  (rf_pred = IfcWall)
Vote distribution across all 200 trees:
IfcWall                75
IfcRoof                39
IfcPlate               23
IfcSanitaryTerminal    16
IfcSlab                16
IfcRailing             13
IfcWindow               9
IfcColumn               4
IfcStairFlight          3
IfcCovering             1
IfcDoor                 1

[1] ADEM / 2hB8hTwIX4p9nI2Iiu0Imc  (rf_pred = IfcWall)
Vote distribution across all 200 trees:
IfcWall                89
IfcRoof                53
IfcSlab                23
IfcPlate               12
IfcWindow               7
IfcStairFlight          7
IfcRailing              4
IfcColumn               2
IfcCovering             1
IfcSanitaryTerminal     1
IfcDoor                 1

[2] ADEM / 2wyUv$j4H8IAbe2YmTbaio  (rf_pred = IfcWall)
Vote distribution across all 200 trees:
IfcWall                88
IfcSanitaryTerminal    31
IfcRoof  

Conclusion: `IfcPlate` is often misclassified as `IfcRoof` or `IfcWall`. Geometrically `IfcPlate` is close to both, it typically represents an outdoor covering above walls, which makes the three element types hard to separate based on geometry alone, as the misclassifications above show.

### 3.2. Analyse label "ifc_entity" and target class "IfcWindow"

In [21]:
results["rf"]["label_ifc_entity"]["IfcWindow"] = analyze_rf_misses(rf, full_table, rf_features, "label_ifc_entity", "IfcWindow", LABEL_COLS)

Found 110 rows where RF misclassified true class 'IfcWindow' on label_ifc_entity

[0] ADEM / 1vFwq7a5r5sOdw0mjQoZGm  (rf_pred = IfcDoor)
Vote distribution across all 200 trees:
IfcDoor                121
IfcWindow               59
IfcWall                 15
IfcRailing               3
IfcSanitaryTerminal      2

[1] ADEM / 1Eai3ZjvT5bxcUpwYWgKYh  (rf_pred = IfcDoor)
Vote distribution across all 200 trees:
IfcDoor                120
IfcWindow               59
IfcWall                 16
IfcRailing               3
IfcSanitaryTerminal      2

[2] ADEM / 09EgNOhH0zriATwcngGc_M  (rf_pred = IfcDoor)
Vote distribution across all 200 trees:
IfcDoor                125
IfcWindow               57
IfcWall                 13
IfcSanitaryTerminal      2
IfcRailing               2
IfcSlab                  1

[3] ADEM / 3h0$8WOHjij7kJGCUlaRJQ  (rf_pred = IfcDoor)
Vote distribution across all 200 trees:
IfcDoor                125
IfcWindow               57
IfcWall                 13
IfcSanitaryTerminal   

Conclusion: The class `IfcWindow` is often misclassified as `IfcWall` or `IfcDoor`. A likely cause is that the material `glass` is currently not extracted from the geometry, adding it would change the feature space and could improve the model's performance. Since these three element types share very similar geometry, relational or contextual features could further help: windows and doors are typically embedded in walls, and that spatial relationship could be extracted via the IfcOpenShell library.

In [22]:
# save the first 5 IfcWindow misclassification examples as JSON for the report table
EXAMPLES_PATH = "misclassification_examples_ifc_window.json"

window_examples = []
for entry in results["rf"]["label_ifc_entity"]["IfcWindow"][:4]:
    top_4 = list(entry["votes"].items())[:4]
    window_examples.append({
        "project_code": entry["project_code"],
        "guid": entry["guid"],
        "rf_pred": entry["rf_pred"],
        "top_5_votes": [[cls, int(n)] for cls, n in top_4],
    })

Path(EXAMPLES_PATH).write_text(json.dumps(window_examples, indent=2))
print(f"saved {len(window_examples)} examples to: {Path(EXAMPLES_PATH).resolve()}")

saved 4 examples to: /Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/code/5_evaluation/5_4_misclassification_analysis/misclassification_examples_ifc_window.json


### 3.3. Analyse label "predefined_type" and target class "COLUMN"

In [23]:
results["rf"]["label_predefined_type"]["COLUMN"] = analyze_rf_misses(rf, full_table, rf_features, "label_predefined_type", "COLUMN", LABEL_COLS)

Found 5 rows where RF misclassified true class 'COLUMN' on label_predefined_type



[0] ADEM / 1XeUUnGW1A591wHmPdrH90  (rf_pred = NOTDEFINED)
Vote distribution across all 200 trees:
NOTDEFINED      92
SOLIDWALL       57
COLUMN          31
PLUMBINGWALL    16
PARTITIONING     2
BASESLAB         1
GUARDRAIL        1

[1] ADEM / 2xVIknWIj6Nepc1hRyicEE  (rf_pred = NOTDEFINED)
Vote distribution across all 200 trees:
NOTDEFINED      95
COLUMN          48
SOLIDWALL       37
PLUMBINGWALL    15
PARTITIONING     4
BASESLAB         1

[2] ADEM / 1G1Ix$IJDFkvXu5g5E0lUY  (rf_pred = SOLIDWALL)
Vote distribution across all 200 trees:
NOTDEFINED      107
SOLIDWALL        90
COLUMN            2
PARTITIONING      1

[3] ADEM / 3uUfBaYKr0Axv4gIKVul$t  (rf_pred = NOTDEFINED)
Vote distribution across all 200 trees:
NOTDEFINED      83
COLUMN          67
SOLIDWALL       39
PLUMBINGWALL     6
PARTITIONING     4
BASESLAB         1

[4] ADEM / 2y8H0Eo6L9gPJ8941FKMwK  (rf_pred = NOTDEFINED)
Vote distribution across all 200 trees:
NOTDEFINED      116
SOLIDWALL        82
PLUMBINGWALL      1
COLUMN

Conclusion: The class `NOTDEFINED` still confuses the model due to its high variability, it absorbs many heterogeneous elements. In most misclassifications the correct label appears in second place in the vote distribution, which suggests the model has the right answer in its candidate set but is pulled towards `NOTDEFINED` by its broad feature footprint.

## 4. Load XGBoost model for later analysis

In [24]:
# load XGBoost model
xgb_path = find_best_model_path("xgboost", MODELS_DIR)
xgb_full = load_model_pickle(xgb_path)
xgb_features = load_feature_names("xgboost", BEST_RUNS_PATH)

# wrapping a StringToIntWrapper per label
xgb_multi = xgb_full.pipeline.named_steps["model"]
xgb_wrapper = xgb_multi.estimators_[1]
xgb_clf = xgb_wrapper.estimator
booster = xgb_clf.get_booster()
le = xgb_wrapper.le_

print(f"Loaded XGB ({booster.num_boosted_rounds()} boosting rounds) for {EXAMPLE_LABEL}")
print(f"XGB was trained on {len(xgb_features)} features")
print(f"Classes (encoded order): {list(le.classes_)}")


Loaded XGB (1000 boosting rounds) for label_predefined_type
XGB was trained on 35 features
Classes (encoded order): ['BASESLAB', 'COLUMN', 'DOOR', 'FLAT_ROOF', 'FLOOR', 'FLOORING', 'GFA', 'GUARDRAIL', 'INTERNAL', 'NOTDEFINED', 'PARTITIONING', 'PLUMBINGWALL', 'SHEET', 'SOLIDWALL', 'TOILETPAN', 'WASHHANDBASIN', 'WINDOW']


### 4.1. Analyse label "predefined_type" and target class "PLUMBINGWALL"

In [25]:
results["xgboost"]["label_predefined_type"]["PLUMBINGWALL"] = analyze_xgb_misses(xgb_multi, full_table, xgb_features, "label_predefined_type", "PLUMBINGWALL", LABEL_COLS, len(xgb_features))
plot_xgb_misses_contributions(xgb_multi, full_table, xgb_features, "label_predefined_type", "PLUMBINGWALL", LABEL_COLS, len(xgb_features))

[0] ADEM / 1PBKQe_br5x8QPrclJIo4G  (xgb_pred = SOLIDWALL)

Top features pushing logit of WRONG class 'SOLIDWALL':
                  feature     value  contribution
               aabb_len_z  2.280000     -0.578507
           aabb_ratio_x_y 29.888984      0.506986
           aabb_ratio_z_y 12.666667      0.362310
            tfbb_extent_1  2.873597     -0.190015
           aabb_ratio_z_x  0.423790      0.163877
            mat_allgemein  1.000000      0.159934
               aabb_max_z  2.280000     -0.154256
               aabb_max_x  5.380017      0.151701
            mat_aluminium  0.000000     -0.147793
               aabb_len_y  0.180000      0.140685
      geom_ratio_area_vol 12.360763      0.125149
            mat_backstein  0.000000     -0.116712
       topo_avg_face_area  1.363824     -0.096887
                 mat_gips  0.000000      0.092434
         geom_compactness  0.300509     -0.089651
         geom_layer_count  1.000000      0.080855
                mat_beton  0.000000 

Conclusion: The geometry ratios of the misclassified `PLUMBINGWALL` elements are closer to the typical `SOLIDWALL` geometry, and the materials are brick and aluminium — neither fits a typical `PLUMBINGWALL`. This points to a likely label error in the ground truth: the XGBoost prediction may actually be correct, and the dataset label wrong.

### 4.2. Analyse label "ifc_entity" and target class "IfcCovering"

In [26]:
results["xgboost"]["label_ifc_entity"]["IfcCovering"] = analyze_xgb_misses(xgb_multi, full_table, xgb_features, "label_ifc_entity", "IfcCovering", LABEL_COLS, len(xgb_features))
plot_xgb_misses_contributions(xgb_multi, full_table, xgb_features, "label_ifc_entity", "IfcCovering", LABEL_COLS, len(xgb_features), save="xgb_ifc_covering_misses_contributions",chapter="evaluation")

[0] ADEM / 3bVq24CkH18OH96PjhhYKP  (xgb_pred = IfcSlab)

Top features pushing logit of WRONG class 'IfcSlab':
                  feature     value  contribution
               aabb_max_z  0.000000      2.325778
               aabb_len_x  0.330000     -0.466235
                mat_beton  0.000000     -0.405419
           aabb_ratio_z_y  0.021875      0.396651
            tfbb_extent_2  0.070000      0.347167
               aabb_len_y  3.199996      0.279760
               aabb_min_z -0.070000      0.246202
            tfbb_ratio_12  4.714286     -0.172184
       topo_avg_face_area  0.217183      0.126000
         geom_compactness  0.326827      0.122362
   topo_vertex_edge_ratio  0.444444      0.117618
           aabb_ratio_z_x  0.212121      0.114607
           aabb_ratio_x_y  0.103125     -0.092979
         geom_layer_count  1.000000     -0.085732
   topo_unique_edge_count 18.000000     -0.075032
      geom_ratio_area_vol 35.257035      0.069817
          topo_face_count 12.000000     

Conclusion: The mean contributions show that `mat_beton` dominates across the misclassifications. Concrete does not fit the definition of `IfcCovering` and is more typical of an `IfcSlab`, so these are likely label errors in the ground truth where the prediction is actually correct. In other cases the material `mat_allgemein` is correctly set, but the geometry deviates from a typical `IfcCovering`, which likely drives the misclassification.

### 4.3. Analyse label "predefined_type" and target class "BASESLAB" (XGBoost)

In [27]:
results["xgboost"]["label_predefined_type"]["BASESLAB"] = analyze_xgb_misses(xgb_multi, full_table, xgb_features, "label_predefined_type", "BASESLAB", LABEL_COLS, len(xgb_features))
plot_xgb_misses_contributions(xgb_multi, full_table, xgb_features, "label_predefined_type", "BASESLAB", LABEL_COLS, len(xgb_features))

[0] ADEM / 02Huz0QMX7Wvl50833$qbw  (xgb_pred = FLOOR)

Top features pushing logit of WRONG class 'FLOOR':
                  feature     value  contribution
           aabb_ratio_z_x  0.090909      1.555542
               aabb_max_z  0.000000      1.519730
      geom_ratio_area_vol 11.575758      0.573374
           aabb_ratio_z_y  0.066667      0.539106
                mat_beton  1.000000      0.360262
            tfbb_extent_0  3.000000     -0.254083
               aabb_len_x  2.200000      0.237612
                 mat_kies  0.000000      0.121920
               aabb_len_z  0.200000      0.108780
       topo_avg_face_area  1.273333      0.058502
         geom_compactness  0.380841      0.055396
            tfbb_ratio_12 11.000000     -0.047779
   topo_vertex_edge_ratio  0.444444      0.045400
               aabb_max_x  2.200000      0.039355
               mat_zement  0.000000      0.035599
            tfbb_extent_2  0.200000     -0.032991
         geom_layer_count  1.000000     -0.0

Conclusion: The misclassified elements have a higher layer count than correctly classified ones, a `BASESLAB` should only have a single layer, so the extra layers likely point to a modelling error in the ground truth rather than a model error. The same goes for cases where the material is `kies` (gravel), which is not consistent with a `BASESLAB` and suggests a labelling issue.

## 5. Save all results as JSON for the report

In [28]:
# save results as json
Path(RESULTS_PATH).write_text(json.dumps(results, indent=2, default=str))
print(f"saved to: {Path(RESULTS_PATH).resolve()}")

saved to: /Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/code/5_evaluation/5_4_misclassification_analysis/misclassification_analysis_results.json
